# Exploration des données CelesTrak

Objectif : comprendre la structure des données avant de calculer les altitudes.

1. Chargement
2. Contrôle qualité
3. Fraîcheur des données
4. Sélection des objets LEO
5. Altitude par la loi de Kepler (vérification, cf. D1 dans `decisions.md`)
6. Altitude par SGP4 (méthode principale) : test sur l'ISS, sphère et ellipsoïde WGS84

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sgp4 import omm
from sgp4.api import Satrec, jday

MU = 398600.4418      # constante gravitationnelle de la Terre (km³/s²)
R_TERRE = 6378.137    # rayon équatorial de la Terre (km)

## 1. Chargement

Dernier fichier téléchargé par le script de collecte.

In [ ]:
fichiers = sorted(Path("../data/raw").glob("visual_*.json"))
chemin = fichiers[-1]
df = pd.read_json(chemin)
chemin

In [ ]:
df.shape

In [ ]:
df.head()

## 2. Contrôle qualité

Doublons et valeurs physiquement impossibles.

In [ ]:
print("Doublons :", df["NORAD_CAT_ID"].duplicated().sum())
print("Excentricités hors de [0, 1[ :", (~df["ECCENTRICITY"].between(0, 1, inclusive="left")).sum())
print("Mouvements moyens négatifs ou nuls :", (df["MEAN_MOTION"] <= 0).sum())

## 3. Fraîcheur des données

Âge de chaque mesure (`EPOCH`) par rapport à maintenant, en jours.

In [ ]:
df["EPOCH"] = pd.to_datetime(df["EPOCH"], utc=True)
maintenant = pd.Timestamp.now(tz="UTC")
df["AGE_JOURS"] = (maintenant - df["EPOCH"]).dt.total_seconds() / 86400
df["AGE_JOURS"].describe()

## 4. Sélection des objets LEO

Mouvement moyen > 11,25 tours/jour, soit une période inférieure à 128 min.

In [ ]:
leo = df[df["MEAN_MOTION"] > 11.25]
len(leo)

In [ ]:
leo["AGE_JOURS"].mean()

In [ ]:
# mesure la plus ancienne
leo[leo["AGE_JOURS"] == leo["AGE_JOURS"].max()]

## 5. Altitude par la loi de Kepler (méthode de vérification)

In [ ]:
n = df["MEAN_MOTION"] * 2 * np.pi / 86400      # tours/jour → rad/s
a = (MU / n**2) ** (1 / 3)                     # demi-grand axe (km)
df["PERIGEE_KEPLER_KM"] = a * (1 - df["ECCENTRICITY"]) - R_TERRE
df["APOGEE_KEPLER_KM"] = a * (1 + df["ECCENTRICITY"]) - R_TERRE
df[["PERIGEE_KEPLER_KM", "APOGEE_KEPLER_KM"]].describe()

Vérification sur l'ISS (NORAD 25544) : périgée attendu vers 415 km.

In [ ]:
df.loc[df["NORAD_CAT_ID"] == 25544, ["OBJECT_NAME", "PERIGEE_KEPLER_KM", "APOGEE_KEPLER_KM"]]

## 6. Altitude par SGP4 (méthode principale) : test sur l'ISS

Propagation de l'ISS sur une orbite complète à partir de maintenant, altitude au-dessus de la sphère et de l'ellipsoïde WGS84, puis comparaison avec Kepler.

In [ ]:
objets = json.loads(chemin.read_text())
iss_omm = [o for o in objets if o["OBJECT_NAME"] == "ISS (ZARYA)"][0]

sat = Satrec()
omm.initialize(sat, iss_omm)

### Altitude au-dessus de l'ellipsoïde WGS84

La Terre est aplatie aux pôles : l'altitude au-dessus d'une sphère varie avec la latitude. On calcule donc l'altitude géodésique, comme prévu dans D1.

La position `r` donnée par SGP4 est dans le repère TEME, qui ne diffère du repère terrestre que par une rotation autour de l'axe des pôles : la latitude, et donc l'altitude, restent valables.

In [ ]:
A_WGS84 = 6378.137                 # rayon équatorial (km)
F_WGS84 = 1 / 298.257223563        # aplatissement
E2_WGS84 = F_WGS84 * (2 - F_WGS84) # excentricité au carré de l'ellipsoïde


def altitude_geodesique(r):
    """Altitude (km) au-dessus de l'ellipsoïde WGS84, à partir de la position r (km)."""
    x, y, z = r
    p = np.hypot(x, y)                         # distance à l'axe des pôles
    lat = np.arctan2(z, p * (1 - E2_WGS84))    # première estimation de la latitude
    for _ in range(5):                         # quelques itérations suffisent
        sin_lat = np.sin(lat)
        n = A_WGS84 / np.sqrt(1 - E2_WGS84 * sin_lat**2)
        h = p * np.cos(lat) + z * sin_lat - A_WGS84 * np.sqrt(1 - E2_WGS84 * sin_lat**2)
        lat = np.arctan2(z, p * (1 - E2_WGS84 * n / (n + h)))
    sin_lat = np.sin(lat)
    return p * np.cos(lat) + z * sin_lat - A_WGS84 * np.sqrt(1 - E2_WGS84 * sin_lat**2)

In [ ]:
maintenant = pd.Timestamp.now(tz="UTC")
periode_min = 1440 / iss_omm["MEAN_MOTION"]
instants = maintenant + pd.to_timedelta(np.linspace(0, periode_min, 200), unit="min")

alt_sphere, alt_wgs84 = [], []
for t in instants:
    jd, fr = jday(t.year, t.month, t.day, t.hour, t.minute, t.second + t.microsecond / 1e6)
    erreur, r, v = sat.sgp4(jd, fr)
    if erreur != 0:
        print(f"Erreur SGP4 (code {erreur}) à {t}")
        continue
    alt_sphere.append(np.linalg.norm(r) - R_TERRE)
    alt_wgs84.append(altitude_geodesique(r))

### Comparaison des méthodes

Kepler suppose une Terre sphérique : il doit être proche de SGP4 sphère, pas de SGP4 WGS84.

In [ ]:
iss = df[df["OBJECT_NAME"] == "ISS (ZARYA)"].iloc[0]
print(f"SGP4, sphère : de {min(alt_sphere):.1f} à {max(alt_sphere):.1f} km")
print(f"SGP4, WGS84  : de {min(alt_wgs84):.1f} à {max(alt_wgs84):.1f} km")
print(f"Kepler       : de {iss['PERIGEE_KEPLER_KM']:.1f} à {iss['APOGEE_KEPLER_KM']:.1f} km")

### Conclusion du test sur l'ISS
- SGP4 (sphère) et Kepler concordent à environ 1,5 km près : la propagation est validée.
- L'altitude WGS84 est plus élevée de 2 à 11 km aux hautes latitudes : la sphère introduit
  une erreur systématique, d'où le choix de WGS84 pour la suite (décision D1).